In [ ]:
import json
import os
import re

import numpy as np

from mp_api.client import MPRester
from pymatgen.io.vasp import Poscar

MP_ID = ''  # Materials Project API key

path_to_folder = 'data-folder'

min_gap   = None  # None keeps metals (needed for metal-insulator transitions); 0.5 reproduces the old insulators-only database
fetch_dos = True  # also download the density of states (needed for the electronic free energy)

In [ ]:
def generate_POSCAR(path_to_folder, doc, attributes=None, dos=None):
    """A folder is created, and a POSCAR file is written from the data extracted from Materials Project.

    Args:
        path_to_folder (str):                            The directory path for storing the generated files.
        doc            (MaterialsProject data structure: The document or structure to generate the POSCAR from.
        attributes     (dict or None, optional):         Attributes to the POSCAR (formation energy, band-gap, etc).
        dos            (pymatgen Dos or None):           Density of states, saved as json file `dos`.

    Returns:
        None
    """

    # Folder with the loaded elements
    if not os.path.exists(path_to_folder):
        os.mkdir(path_to_folder)
    
    # Folder with the specific element
    path_to_folder = f'{path_to_folder}/{doc.formula_pretty}'
    if not os.path.exists(path_to_folder):
        os.mkdir(path_to_folder)

    # Folder with the specific symmetry of the element
    # The material id is part of the name: two polymorphs with the same space group no longer overwrite each other
    symmetry_folder = re.sub('/', '-', doc.symmetry.symbol) + f'_{doc.material_id}'
    path_to_folder  = f'{path_to_folder}/{symmetry_folder}'
    if not os.path.exists(path_to_folder):
        os.mkdir(path_to_folder)
    
    # Writing attributes per atom
    if attributes is not None:
        for key, value in attributes.items():
            print(key, value)
            try:
                np.savetxt(f'{path_to_folder}/{key}', [value])
            except Exception as error:
                print(f'{doc.material_id}: could not write {key} ({error})')
    
    # Writing POSCAR (pymatgen groups the sites by species; the previous writer used composition_concentration, which was
    # not defined anywhere, and wrote the sites in structure order, which is wrong if species are not contiguous)
    Poscar(doc.structure, comment=f'{doc.formula_pretty} ({doc.material_id})').write_file(f'{path_to_folder}/POSCAR')

    # Writing the density of states (spin channels summed)
    if dos is not None:
        with open(f'{path_to_folder}/dos', 'w') as dos_file:
            json.dump({'energies':  [float(e) for e in dos.energies],
                       'densities': np.sum([np.asarray(d) for d in dos.densities.values()], axis=0).tolist(),
                       'efermi':    float(dos.efermi),
                       'natoms':    len(doc.structure)}, dos_file)


In [ ]:
# Earth-abundant materials

elements = ['H',  'He', 'Li', 'Be', 'B',  'C',  'N',  'O',  'F',  'Ne', 'Na', 'Mg', 'Al', 'Si', 'P',  'S',  'Cl',
            'Ar', 'K',  'Ca', 'Ti', 'V',  'Cr', 'Mn', 'Fe', 'Co', 'Ni', 'Cu', 'Zn', 'Ga', 'Ge', 'As', 'Se', 'Br',
            'Kr', 'Sr', 'Y',  'Zr', 'Nb', 'Mo', 'Rh', 'Ag', 'Cd', 'In', 'Sn', 'Sb', 'Te', 'I',  'Ba', 'La', 'Ce',
            'Pr', 'Nd', 'Sm', 'Eu', 'Gd', 'Tb', 'Dy', 'Ho', 'Er', 'Yb', 'Lu', 'Hf', 'Ta', 'W',  'Hg', 'Pb', 'Bi',
            'At', 'Rn', 'Fr', 'Ra', 'Pa', 'U',  'Es', 'Fm', 'Md', 'No', 'Lr', 'Rf', 'Db', 'Sg', 'Bh', 'Hs', 'Mt',
            'Ds', 'Rg', 'Cn', 'Nh', 'Fl', 'Mc', 'Lv', 'Ts', 'Og', 'Cs', 'Pd', 'Au', 'Xe']

exclude_elements = ['Po', 'Ac', 'Tc', 'Cf', 'Bk', 'Cm', 'Pu', 'Am', 'Np', 'Pm',
                    'Ir', 'Pt', 'Rb', 'Os', 'Ru', 'Tl', 'Sc', 'Re', 'Th', 'Tm']

fields = ['material_id', 'formula_pretty', 'structure', 'symmetry', 'energy_per_atom', 'band_gap']

seen = set()  # a compound is returned once for each of its elements
for element in elements:
    print(element)
    with MPRester(MP_ID) as mpr:
        docs = mpr.materials.summary.search(elements=[element],
                                  exclude_elements=exclude_elements,
                                  fields=fields)

        for i in range(len(docs)):
            doc = docs[i]
            if doc.material_id in seen:
                continue
            seen.add(doc.material_id)
            try:
                if min_gap is None or doc.band_gap > min_gap:
                    dos = None
                    if fetch_dos:
                        try:
                            dos = mpr.get_dos_by_material_id(doc.material_id)
                        except Exception as error:
                            print(f'{doc.material_id}: no DOS ({error})')

                    generate_POSCAR(path_to_folder, doc,
                                    attributes={
                                        'EPA':     doc.energy_per_atom,
                                        'bandgap': doc.band_gap
                                        },
                                    dos=dos
                                    )
            except Exception as error:
                print(f'{doc.material_id}: {error}')